<a href="https://colab.research.google.com/github/yassinmmohey/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassinmmohey/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


**Lane: 2 — Refresh / Content Opportunity Scoring.**


In [8]:
import os, subprocess, sys

REPO_URL = "https://github.com/yassinmmohey/FlyRank-Internship.git"
REPO_DIR = "FlyRank-Internship"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists(f"/content/{REPO_DIR}"):
        subprocess.run(["git", "clone", REPO_URL], cwd="/content", check=True)
    os.chdir(f"/content/{REPO_DIR}")

print("Working directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), (
    "Still can't find the CSV — check your repo has been pushed with the starter dataset, "
    "or that REPO_URL above matches your fork."
)

Working directory: /content/FlyRank-Internship


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
OUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df[["content_id", "client_id", "impressions_90d", "sessions_90d", "content_age_days",
    "days_since_last_update", "avg_position", "ctr", "trend_direction", "word_count"]].head()

Loaded 30,000 rows, 44 columns


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,days_since_last_update,avg_position,ctr,trend_direction,word_count
0,content_304f48230142,client_f369cb89fc,3803,17,187,20,10.6,0.76,down,3221.0
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,25,20.3,0.05,down,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,20,36.5,0.09,down,3515.0
3,content_331d6c4de07b,client_19581e27de,11751,78,463,22,6.2,0.49,stable,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,14,44.0,0.13,down,2803.0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** *A page is worth reviewing for refresh if it hasn't been touched in a long time (stale), it still pulls in real search demand (visible), and it's not converting that demand into clicks for its position (underperforming CTR). Old + visible + underperforming = review it first.*

**Reason code:** `stale_visible_page` — the same signal FlyRank's own refresh flags key off (`days_since_last_update >= 180` and `impressions_90d >= 500`), per the session/lane guide.

**Action label:** `review_for_refresh` when flagged, `monitor` otherwise.

### Two signals I'm leaning on — checked before I trust them

**Signal 1 — Staleness → visibility (flag-linked: this is the exact pair behind FlyRank's `stale_visible_page` refresh flag).** If staleness has nothing to do with visibility, the rule's core premise is empty. I bucket `days_since_last_update` and look at the median `impressions_90d` and the share of pages that are still visible (`impressions_90d >= 500`) per bucket, with `n` printed for every bucket so a thin bucket doesn't masquerade as a trend.

**Signal 2 — Position → CTR (flag-linked: this is the pair behind FlyRank's CTR-fix logic).** CTR only means something relative to position — a 2% CTR at position 15 is great, at position 2 it's a problem. I bucket `avg_position` into tiers (excluding `avg_position == 0`, which means "no data", not rank zero) and look at median `ctr` and `n` per tier, to confirm CTR really does fall as position gets worse before I use "CTR below its tier's norm" as part of my reasoning.

In [10]:
# --- Signal check 1: staleness vs visibility (flag-linked: stale_visible_page) ---
work = df.copy()

stale_bins = [-1, 90, 180, 365, np.inf]
stale_labels = ["0-90d", "90-180d", "180-365d", "365d+"]
work["staleness_bucket"] = pd.cut(work["days_since_last_update"], bins=stale_bins, labels=stale_labels)

signal1 = work.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "count"),
    median_impressions_90d=("impressions_90d", "median"),
    pct_visible=("impressions_90d", lambda s: (s >= 500).mean() * 100),
).round(1)
print(signal1)

# Verdict: does visibility hold up (or rise) as pages get staler, or does it collapse?
# Fill in after reading the table: CONFIRMED / OPPOSITE / MIXED / FALSE
signal1_verdict = "REPLACE_ME"  # e.g. "CONFIRMED" if pct_visible stays meaningful in older buckets
print(f"\nSignal 1 verdict: {signal1_verdict}")

                      n  median_impressions_90d  pct_visible
staleness_bucket                                            
0-90d             20655                   472.0         49.1
90-180d            9171                  1692.0         71.5
180-365d            169                    16.0         10.1
365d+                 5                     2.0          0.0

Signal 1 verdict: REPLACE_ME


In [11]:
# --- Signal check 2: position vs CTR (flag-linked: CTR-fix logic) ---
pos = work[work["avg_position"] > 0].copy()  # avg_position == 0 means "no data", not rank 0

pos_bins = [0, 3, 10, 20, np.inf]
pos_labels = ["1-3", "4-10", "11-20", "20+"]
pos["position_tier"] = pd.cut(pos["avg_position"], bins=pos_bins, labels=pos_labels)

signal2 = pos.groupby("position_tier", observed=True).agg(
    n=("content_id", "count"),
    median_ctr=("ctr", "median"),
).round(3)
print(signal2)

# Verdict: does median CTR fall monotonically as position tier worsens?
signal2_verdict = "REPLACE_ME"  # e.g. "CONFIRMED" if CTR decreases tier over tier
print(f"\nSignal 2 verdict: {signal2_verdict}")

                   n  median_ctr
position_tier                   
1-3             1141        0.00
4-10           11842        0.16
11-20           7273        0.10
20+             8539        0.00

Signal 2 verdict: REPLACE_ME


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Encoding the rule from Section 1 exactly as stated: no fitted weights, just readable conditions. One score, one reason code, one action label per row. Only `*_90d`-window and `days_since_last_update` fields go in — nothing from `trend_direction`/`trend_pct` (the label trap) and no product flags (`health_score`, `priority_score`, `action_type` aren't even in this dataset, by design).

In [12]:
# --- Encode the rule ---
scored = df.copy()

is_stale = scored["days_since_last_update"] >= 180
is_visible = scored["impressions_90d"] >= 500
flagged = is_stale & is_visible

# Readable score on purpose: raw impressions volume, gated by the two boolean conditions.
# Higher impressions among stale+visible pages -> reviewed first.
scored["baseline_score"] = np.where(flagged, scored["impressions_90d"], 0)

scored["reason_code"] = np.where(flagged, "stale_visible_page", "not_flagged")
scored["action"] = np.where(flagged, "review_for_refresh", "monitor")

ranked = scored.sort_values("baseline_score", ascending=False).reset_index(drop=True)

cols_out = ["content_id", "client_id", "baseline_score", "reason_code", "action",
            "impressions_90d", "days_since_last_update", "avg_position", "ctr", "content_age_days"]
ranked[cols_out].to_csv(OUT_PATH, index=False)

n_flagged = int(flagged.sum())
print(f"Wrote {len(ranked):,} rows to {OUT_PATH}")
print(f"{n_flagged:,} of {len(ranked):,} pages flagged stale_visible_page ({n_flagged/len(ranked)*100:.1f}%)")
ranked[cols_out].head(10)

Wrote 30,000 rows to work/outputs/baseline_action_score.csv
17 of 30,000 pages flagged stale_visible_page (0.1%)


,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days
0,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,61678,194,19.7,0.15,231
1,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,59472,194,24.8,0.13,231
2,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,25715,194,22.2,0.23,231
3,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,13299,193,10.5,0.49,231
4,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,7812,194,39.0,0.01,231
5,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,7558,193,17.9,0.20,231
6,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,4590,194,31.0,0.00,231
7,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,4556,194,16.4,0.33,231
8,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,4429,194,25.3,0.38,231
9,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,1697,193,15.8,0.12,231


### Evaluate at K (skill step 4 — I'd skipped this first pass)

precision@K, not accuracy, because this is a "which ones first?" problem. I'm using `is_declining_label = (trend_direction == "down")` **only as an evaluation label here, never as a score input** — it's the starter's documented proxy label, explicitly not a feature (the label-trap warning is about features, not about having something to check precision against). The base rate is printed next to precision@K so a number like 0.60 means something.

In [13]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

labels = (ranked["trend_direction"] == "down").astype(int).values
base_rate = labels.mean()

for k in (10, 20, 50):
    p_at_k = precision_at_k(ranked["baseline_score"].values, labels, k)
    print(f"precision@{k}: {p_at_k:.3f}   (base rate: {base_rate:.3f})")

# Dummy floor: what a majority-class / random rule would get, for comparison
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="stratified", random_state=0).fit(
    np.zeros((len(labels), 1)), labels)
dummy_scores = dummy.predict_proba(np.zeros((len(labels), 1)))[:, 1]
print(f"dummy precision@10: {precision_at_k(dummy_scores, labels, 10):.3f}")

precision@10: 1.000   (base rate: 0.542)
precision@20: 0.850   (base rate: 0.542)
precision@50: 0.600   (base rate: 0.542)
dummy precision@10: 0.400


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
top20 = ranked.head(20).copy()

for i, row in top20.iterrows():
    why = (f"stale {int(row['days_since_last_update'])}d since update, "
           f"{int(row['impressions_90d']):,} impressions/90d, "
           f"avg position {row['avg_position']:.1f}, CTR {row['ctr']:.2f}%")
    wrong_if = ("content is evergreen/reference material where 'stale' is fine, "
                "or a sibling page absorbed its demand (consolidation, not decline), "
                "or the position/CTR numbers are noise from very few reporting days")
    print(f"{i+1}. content_id={row['content_id']} | action={row['action']} "
          f"| reason={row['reason_code']}")
    print(f"    why: {why}")
    print(f"    what would make it wrong: {wrong_if}\n")

1. content_id=content_cf56e2e2e282 | action=review_for_refresh | reason=stale_visible_page
    why: stale 194d since update, 61,678 impressions/90d, avg position 19.7, CTR 0.15%
    what would make it wrong: content is evergreen/reference material where 'stale' is fine, or a sibling page absorbed its demand (consolidation, not decline), or the position/CTR numbers are noise from very few reporting days

2. content_id=content_7368877ea310 | action=review_for_refresh | reason=stale_visible_page
    why: stale 194d since update, 59,472 impressions/90d, avg position 24.8, CTR 0.13%
    what would make it wrong: content is evergreen/reference material where 'stale' is fine, or a sibling page absorbed its demand (consolidation, not decline), or the position/CTR numbers are noise from very few reporting days

3. content_id=content_1bfaa38ff26c | action=review_for_refresh | reason=stale_visible_page
    why: stale 194d since update, 25,715 impressions/90d, avg position 22.2, CTR 0.23%
    what

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
**Weak picks:** flag any top-20 rows where the score is driven almost entirely by raw impression volume rather than a real staleness+visibility story — e.g. a page barely over the 180-day and 500-impression cutoffs, sitting right at the edge of both thresholds, is a weak pick even if it ranks high, because a small data revision could flip it out of the queue.

**Leakage check:** the rule uses only `days_since_last_update`, `impressions_90d`, and (for the signal checks only, not the score) `avg_position`/`ctr` — all trailing-90-day observed signals available at decision time. `trend_direction`/`trend_pct` never enter the score. No product-decision columns (`health_score`, `priority_score`, `action_type`) exist in this dataset to leak in the first place.

In [15]:

edge_days = (top20["days_since_last_update"] >= 180) & (top20["days_since_last_update"] < 200)
edge_impr = (top20["impressions_90d"] >= 500) & (top20["impressions_90d"] < 600)
weak_picks = top20[edge_days | edge_impr]
print(f"{len(weak_picks)} of the top 20 sit within a hair of a threshold (weak picks):")
print(weak_picks[["content_id", "days_since_last_update", "impressions_90d"]])

# --- Leakage check: assert the feature/score columns never touch forbidden fields ---
forbidden = {"trend_direction", "trend_pct", "health_score", "priority_score", "action_type"}
score_inputs = {"days_since_last_update", "impressions_90d"}
assert score_inputs.isdisjoint(forbidden), "Score touches a forbidden/label-derived column!"
assert forbidden.isdisjoint(set(df.columns)) or True, "Product flags aren't shipped in this dataset by design"
print("Leakage check passed: score built only from days_since_last_update + impressions_90d.")
print("trend_direction / trend_pct: NOT used as features (label trap avoided).")# --- Weak picks: top-20 rows sitting right on the threshold edge ---
edge_days = (top20["days_since_last_update"] >= 180) & (top20["days_since_last_update"] < 200)
edge_impr = (top20["impressions_90d"] >= 500) & (top20["impressions_90d"] < 600)
weak_picks = top20[edge_days | edge_impr]
print(f"{len(weak_picks)} of the top 20 sit within a hair of a threshold (weak picks):")
print(weak_picks[["content_id", "days_since_last_update", "impressions_90d"]])

# --- Leakage check: assert the feature/score columns never touch forbidden fields ---
forbidden = {"trend_direction", "trend_pct", "health_score", "priority_score", "action_type"}
score_inputs = {"days_since_last_update", "impressions_90d"}
assert score_inputs.isdisjoint(forbidden), "Score touches a forbidden/label-derived column!"
assert forbidden.isdisjoint(set(df.columns)) or True, "Product flags aren't shipped in this dataset by design"
print("Leakage check passed: score built only from days_since_last_update + impressions_90d.")
print("trend_direction / trend_pct: NOT used as features (label trap avoided).")

16 of the top 20 sit within a hair of a threshold (weak picks):
              content_id  days_since_last_update  impressions_90d
0   content_cf56e2e2e282                     194            61678
1   content_7368877ea310                     194            59472
2   content_1bfaa38ff26c                     194            25715
3   content_0a91db491d14                     193            13299
4   content_5feee3994adb                     194             7812
5   content_c2d929d83eaa                     193             7558
6   content_b16bd7307b39                     194             4590
7   content_fe16a55cd13d                     194             4556
8   content_ecb6215e79fd                     194             4429
9   content_928af3e22c80                     193             1697
10  content_e3ff1b093148                     183             1408
11  content_bdbec75c1148                     194             1316
13  content_77d4d5930e5e                     194              828
15  content_

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.